# Práctica de PySpark en Databricks

Notebook de práctica para acompañar las clases de PySpark. Está pensado para Databricks Free Edition, así que todo corre en serverless.

Son 10 módulos y un desafío al final. En cada módulo hay una explicación corta con un ejemplo y luego los ejercicios. Debajo de cada ejercicio hay una celda que comprueba el resultado: si imprime "Correcto", pasa al siguiente; si sale un AssertionError, lee el mensaje y revisa tu código.

Antes de empezar:
1. Conecta el notebook a Serverless (selector de cómputo, arriba a la derecha).
2. Ejecuta las celdas en orden con Shift + Enter, porque varios ejercicios usan resultados de los anteriores.
3. Vamos a usar la tabla `samples.nyctaxi.trips` (viajes de taxi en Nueva York), las tablas de `samples.tpch` y algunos DataFrames pequeños creados a mano.

Nota: en Free Edition no hay API de RDD ni `sparkContext` (serverless usa Spark Connect), así que trabajamos solo con DataFrames y SQL. La variable `spark` ya viene creada.

## 0. Preparación
Importamos lo que vamos a usar en todo el notebook y definimos dónde se guardan las tablas. En Free Edition ya existen el catálogo `workspace` y el esquema `default`, y tienes permiso de escritura ahí.

In [ ]:
from datetime import date
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

CATALOGO = "workspace"
ESQUEMA = "default"

def ok(mensaje="Correcto"):
    print(mensaje)

print("Versión de Spark:", spark.version)

## Módulo 1. Crear DataFrames

Un DataFrame es básicamente una tabla distribuida: filas y columnas con nombre y tipo (eso es el esquema). La forma más simple de crear uno es desde una lista de Python con `spark.createDataFrame`. Los tipos los puede inferir Spark o los podemos declarar nosotros con `StructType`.

In [ ]:
datos = [("Ana", "Ventas", 3200), ("Luis", "TI", 4100)]
df_ejemplo = spark.createDataFrame(datos, ["nombre", "area", "sueldo"])

df_ejemplo.show()
df_ejemplo.printSchema()   # tipos inferidos: string, string, long

### Ejercicio 1.1: DataFrame con esquema explícito
Crea el DataFrame `empleados` con los datos de abajo usando un esquema explícito con estos campos:

| campo | tipo |
|---|---|
| `nombre` | `StringType()` |
| `area` | `StringType()` |
| `sueldo` | `IntegerType()` |
| `fecha_ingreso` | `DateType()` |

In [ ]:
datos_empleados = [
    ("Ana",   "Ventas",   3200, date(2019, 3, 1)),
    ("Luis",  "TI",       4100, date(2020, 7, 15)),
    ("Marta", "TI",       4800, date(2017, 1, 10)),
    ("Jorge", "Ventas",   2900, date(2022, 9, 5)),
    ("Sofía", "Finanzas", 3900, date(2018, 11, 20)),
    ("Pedro", "Finanzas", 3500, date(2021, 5, 3)),
]


empleados.printSchema()

In [ ]:
# comprobación
assert empleados.count() == 6, "Deberías tener 6 filas"
assert dict(empleados.dtypes) == {"nombre": "string", "area": "string", "sueldo": "int", "fecha_ingreso": "date"}, \
    f"Revisa los tipos: {empleados.dtypes}"
ok()

### Ejercicio 1.2: filas y columnas
Guarda en `n_filas` el número de filas de `empleados` y en `n_columnas` el número de columnas.

Pista: `count()` y el atributo `.columns`.

In [ ]:
print(n_filas, n_columnas)

In [ ]:
# comprobación
assert n_filas == 6 and n_columnas == 4, "Esperaba 6 filas y 4 columnas"
ok()

## Módulo 2. Explorar una tabla real

Con `spark.read.table("catalogo.esquema.tabla")` leemos una tabla de Unity Catalog. Vamos a usar `samples.nyctaxi.trips`, que ya viene en el workspace.

`display()` muestra una tabla interactiva (desde ahí también se pueden hacer gráficos) y `show()` imprime texto plano.

In [ ]:
trips = spark.read.table("samples.nyctaxi.trips")
display(trips.limit(5))

### Ejercicio 2.1: esquema y columnas
Imprime el esquema de `trips` y guarda la lista con los nombres de las columnas en `columnas`.

In [ ]:
print(columnas)

In [ ]:
# comprobación
assert isinstance(columnas, list) and "fare_amount" in columnas and "pickup_zip" in columnas
ok()

### Ejercicio 2.2: cantidad de viajes
Guarda en `n_viajes` el número total de filas de `trips`.

In [ ]:
print(f"Hay {n_viajes:,} viajes")

In [ ]:
# comprobación
assert isinstance(n_viajes, int) and n_viajes > 0
ok()

### Ejercicio 2.3: estadísticas descriptivas
Crea un DataFrame `estadisticas` con el conteo, media, desviación estándar, mínimo y máximo de `trip_distance` y `fare_amount`.

Pista: hay un método de DataFrame que hace justo esto, parecido al `.describe()` de pandas.

In [ ]:
display(estadisticas)

In [ ]:
# comprobación
assert estadisticas.columns == ["summary", "trip_distance", "fare_amount"]
ok()

### Ejercicio 2.4: rango de fechas
¿Entre qué fechas están los viajes? Crea `rango` con dos columnas: `desde` (el mínimo de `tpep_pickup_datetime`) y `hasta` (el máximo).

In [ ]:
display(rango)

In [ ]:
# comprobación
fila = rango.first()
assert rango.columns == ["desde", "hasta"] and fila["desde"] <= fila["hasta"]
ok()

## Módulo 3. Seleccionar y filtrar

Lo básico que vamos a usar:

| Para | Se usa |
|---|---|
| elegir columnas | `df.select("a", "b")` |
| filtrar filas | `df.filter(F.col("a") > 10)` (o `where`) |
| combinar condiciones | `&` (y), `\|` (o), `~` (no), cada condición entre paréntesis |
| valor dentro de una lista | `F.col("a").isin(1, 2, 3)` |
| ordenar | `df.orderBy(F.desc("a"))` |
| quitar duplicados | `df.distinct()` |

Ojo: entre columnas no se usa `and` / `or` de Python, se usa `&` y `|`.

In [ ]:
# Ejemplo: viajes de más de 5 millas, ordenados por distancia
display(
    trips.filter(F.col("trip_distance") > 5)
         .select("tpep_pickup_datetime", "trip_distance", "fare_amount")
         .orderBy(F.desc("trip_distance"))
         .limit(5)
)

### Ejercicio 3.1: seleccionar columnas
Crea `viajes_basicos` solo con estas columnas y en este orden: `tpep_pickup_datetime`, `trip_distance`, `fare_amount`, `pickup_zip`.

In [ ]:
display(viajes_basicos.limit(5))

In [ ]:
# comprobación
assert viajes_basicos.columns == ["tpep_pickup_datetime", "trip_distance", "fare_amount", "pickup_zip"]
ok()

### Ejercicio 3.2: dos condiciones
Crea `viajes_largos` con los viajes de más de 10 millas y con tarifa menor a 100.

In [ ]:
print(viajes_largos.count())

In [ ]:
# comprobación
assert viajes_largos.filter((F.col("trip_distance") <= 10) | (F.col("fare_amount") >= 100)).count() == 0, \
    "Hay filas que no cumplen ambas condiciones"
assert viajes_largos.count() == trips.filter("trip_distance > 10 AND fare_amount < 100").count()
ok()

### Ejercicio 3.3: los 5 más caros
Crea `top5_caros` con los 5 viajes de mayor `fare_amount`, ordenados de mayor a menor.

In [ ]:
display(top5_caros)

In [ ]:
# comprobación
maximo = trips.agg(F.max("fare_amount")).first()[0]
filas = top5_caros.collect()   # son 5 filas, el collect no es problema
assert len(filas) == 5 and filas[0]["fare_amount"] == maximo
assert all(filas[i]["fare_amount"] >= filas[i + 1]["fare_amount"] for i in range(4)), "No está ordenado de mayor a menor"
ok()

### Ejercicio 3.4: valores distintos
¿Cuántos códigos postales de origen (`pickup_zip`) distintos hay? Guarda el número en `n_zips`.

In [ ]:
print(n_zips)

In [ ]:
# comprobación
assert n_zips == trips.groupBy("pickup_zip").count().count()
ok()

### Ejercicio 3.5: filtrar por lista
Crea `desde_midtown` con los viajes cuyo `pickup_zip` sea 10001, 10011 o 10019. Usa `isin`.

In [ ]:
display(desde_midtown.groupBy("pickup_zip").count())

In [ ]:
# comprobación
zips = {r["pickup_zip"] for r in desde_midtown.select("pickup_zip").distinct().collect()}
assert zips <= {10001, 10011, 10019} and len(zips) > 0
ok()

## Módulo 4. Columnas nuevas, fechas y condiciones

`withColumn("nombre", expresion)` agrega una columna nueva (o reemplaza una que ya existe). Algunas expresiones que vamos a necesitar:

- Aritmética: `F.col("a") * 2`, `F.round(expr, 2)`
- Fechas y horas: `F.hour`, `F.dayofweek` (1 = domingo, 7 = sábado), `F.to_date`, `F.unix_timestamp` (en segundos)
- Condiciones: `F.when(cond, valor).when(...).otherwise(valor)`
- División segura: `F.try_divide(a, b)`, que devuelve `null` cuando `b` es 0

En serverless está activo el modo ANSI, así que dividir entre cero o hacer un cast inválido lanza error en lugar de devolver `null`. Por eso usamos `try_divide`.

Recuerda que los DataFrames son inmutables: cada `withColumn` devuelve un DataFrame nuevo, hay que guardarlo en una variable.

### Ejercicio 4.1: duración del viaje
A partir de `trips`, crea `viajes_dur` con una columna nueva `duracion_min` con los minutos entre `tpep_pickup_datetime` y `tpep_dropoff_datetime`, redondeados a 1 decimal.

Pista: `(F.unix_timestamp(fin) - F.unix_timestamp(inicio)) / 60`

In [ ]:
display(viajes_dur.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "duracion_min").limit(5))

In [ ]:
# comprobación
esperado = (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
assert "duracion_min" in viajes_dur.columns
assert viajes_dur.filter(F.abs(F.col("duracion_min") - esperado) > 0.06).count() == 0, "Revisa el cálculo de minutos"
ok()

### Ejercicio 4.2: hora y día de la semana
A partir de `viajes_dur`, crea `viajes_tiempo` agregando dos columnas calculadas desde `tpep_pickup_datetime`:
- `hora`: hora del día (0 a 23)
- `dia_semana`: día de la semana como número (1 = domingo, 7 = sábado)

In [ ]:
display(viajes_tiempo.select("tpep_pickup_datetime", "hora", "dia_semana").limit(5))

In [ ]:
# comprobación
r = viajes_tiempo.agg(F.min("hora"), F.max("hora"), F.min("dia_semana"), F.max("dia_semana")).first()
assert 0 <= r[0] and r[1] <= 23 and 1 <= r[2] and r[3] <= 7
assert viajes_tiempo.filter(F.col("hora") != F.hour("tpep_pickup_datetime")).count() == 0
ok()

### Ejercicio 4.3: clasificar con `when`
A partir de `viajes_tiempo`, crea `viajes_tipo` con la columna `tipo_viaje`:
- `"corto"` si `trip_distance` < 2
- `"medio"` si `trip_distance` < 10
- `"largo"` en los demás casos

In [ ]:
display(viajes_tipo.groupBy("tipo_viaje").count())

In [ ]:
# comprobación
mal = viajes_tipo.filter(
    ((F.col("trip_distance") < 2) & (F.col("tipo_viaje") != "corto")) |
    ((F.col("trip_distance") >= 2) & (F.col("trip_distance") < 10) & (F.col("tipo_viaje") != "medio")) |
    ((F.col("trip_distance") >= 10) & (F.col("tipo_viaje") != "largo"))
).count()
assert mal == 0, f"{mal} filas mal clasificadas"
ok()

### Ejercicio 4.4: división segura
A partir de `viajes_tipo`, crea `viajes` con la columna `tarifa_por_milla` = `fare_amount / trip_distance` redondeada a 2 decimales. Este va a ser el DataFrame principal de aquí en adelante.

Hay viajes con distancia 0, así que usa `F.try_divide` para que esos queden en `null` y no salte un error.

In [ ]:
display(viajes.select("trip_distance", "fare_amount", "tarifa_por_milla").limit(10))

In [ ]:
# comprobación
assert viajes.filter((F.col("trip_distance") == 0) & F.col("tarifa_por_milla").isNotNull()).count() == 0
assert viajes.filter((F.col("trip_distance") > 0) & F.col("tarifa_por_milla").isNull()).count() == 0
ok("Correcto. viajes ya tiene duracion_min, hora, dia_semana, tipo_viaje y tarifa_por_milla")

Algo a notar: las celdas de 4.1 a 4.4 todavía no ejecutaron nada, solo describieron transformaciones (evaluación perezosa). Con `explain()` podemos ver el plan que Spark armó y optimizó para `viajes`:

In [ ]:
viajes.explain()

## Módulo 5. Agregaciones

`groupBy(...)` junto con `agg(...)` resume filas por grupo, igual que un `GROUP BY` en SQL.

```python
df.groupBy("columna").agg(
    F.count("*").alias("n"),
    F.avg("otra").alias("media"),
)
```
Las funciones de agregación más comunes son `count`, `sum`, `avg`, `min`, `max` y `countDistinct`. Si necesitas filtrar después de agregar (lo que en SQL sería `HAVING`), aplica `filter` sobre el resultado.

### Ejercicio 5.1: viajes por hora
Crea `por_hora` con el número de viajes por `hora`, ordenado por hora. La columna del conteo se tiene que llamar `n_viajes`.

Cuando salga el resultado del `display`, prueba el botón + > Visualización para armar un gráfico de barras.

In [ ]:
display(por_hora)

In [ ]:
# comprobación
assert por_hora.columns == ["hora", "n_viajes"]
assert por_hora.agg(F.sum("n_viajes")).first()[0] == viajes.count()
horas = [r["hora"] for r in por_hora.collect()]
assert horas == sorted(horas), "Ordena por hora"
ok()

### Ejercicio 5.2: resumen por tipo de viaje
Crea `por_tipo` agrupando por `tipo_viaje`, con estas columnas:
- `n_viajes`: número de viajes
- `tarifa_media`: promedio de `fare_amount`, redondeado a 2 decimales
- `distancia_total`: suma de `trip_distance`, redondeada a 1 decimal

In [ ]:
display(por_tipo)

In [ ]:
# comprobación
assert por_tipo.columns == ["tipo_viaje", "n_viajes", "tarifa_media", "distancia_total"]
assert por_tipo.count() == 3
assert por_tipo.agg(F.sum("n_viajes")).first()[0] == viajes.count()
ok()

### Ejercicio 5.3: top zonas de origen
Crea `top_zips` con los 10 `pickup_zip` que tienen más viajes, considerando solo los que tienen al menos 50 viajes. Columnas: `pickup_zip`, `n_viajes` y `tarifa_media` (redondeada a 2). Ordena de más a menos viajes.

In [ ]:
display(top_zips)

In [ ]:
# comprobación
filas = top_zips.collect()
assert top_zips.columns == ["pickup_zip", "n_viajes", "tarifa_media"]
assert 0 < len(filas) <= 10 and all(r["n_viajes"] >= 50 for r in filas)
assert all(filas[i]["n_viajes"] >= filas[i + 1]["n_viajes"] for i in range(len(filas) - 1))
ok()

## Módulo 6. Joins

`df1.join(df2, condicion, how="tipo")` combina dos DataFrames. Los tipos que más vamos a usar:

| `how=` | Qué devuelve |
|---|---|
| `"inner"` (por defecto) | solo las filas que tienen pareja en ambos lados |
| `"left"` | todas las filas de la izquierda, con `null` donde no hay pareja |
| `"full"` | todas las filas de ambos lados |
| `"left_anti"` | las filas de la izquierda que no tienen pareja (muy útil para control de calidad) |

Si una de las tablas es pequeña, `F.broadcast(tabla)` la copia a cada executor y así se evita el shuffle.

Para los ejercicios vamos a usar esta tabla de códigos postales (simplificada, no están todos):

In [ ]:
zonas = spark.createDataFrame(
    [
        (10001, "Chelsea", "Manhattan"), (10002, "Lower East Side", "Manhattan"),
        (10003, "East Village", "Manhattan"), (10011, "Chelsea", "Manhattan"),
        (10012, "SoHo", "Manhattan"), (10013, "Tribeca", "Manhattan"),
        (10014, "West Village", "Manhattan"), (10016, "Murray Hill", "Manhattan"),
        (10017, "Midtown East", "Manhattan"), (10019, "Midtown West", "Manhattan"),
        (10022, "Midtown East", "Manhattan"), (10023, "Upper West Side", "Manhattan"),
        (10036, "Times Square", "Manhattan"), (11201, "Brooklyn Heights", "Brooklyn"),
        (11211, "Williamsburg", "Brooklyn"), (11101, "Long Island City", "Queens"),
        (11371, "Aeropuerto LaGuardia", "Queens"), (11430, "Aeropuerto JFK", "Queens"),
    ],
    ["zip", "barrio", "distrito"],
)
display(zonas)

### Ejercicio 6.1: inner join
Crea `viajes_zona` uniendo `viajes` con `zonas` donde `pickup_zip == zip` (inner join).

In [ ]:
display(viajes_zona.select("pickup_zip", "barrio", "distrito", "fare_amount").limit(5))

In [ ]:
# comprobación
assert "barrio" in viajes_zona.columns and "distrito" in viajes_zona.columns
assert viajes_zona.filter(F.col("barrio").isNull()).count() == 0, "En un inner join no debería haber barrios nulos"
ok()

### Ejercicio 6.2: left join
Crea `viajes_zona_left` con un left join. Tiene que conservar todos los viajes.

In [ ]:
print(viajes.count(), viajes_zona_left.count())

In [ ]:
# comprobación
assert viajes_zona_left.count() == viajes.count()
ok()

### Ejercicio 6.3: viajes sin zona
Crea `sin_zona` con los viajes cuyo `pickup_zip` no está en `zonas`. Usa `left_anti`.

In [ ]:
print(f"{sin_zona.count():,} viajes sin zona conocida")

In [ ]:
# comprobación
assert sin_zona.columns == viajes.columns, "left_anti solo devuelve las columnas de la izquierda"
assert sin_zona.count() + viajes_zona.count() == viajes.count()
ok()

### Ejercicio 6.4: broadcast + agregación
Crea `por_distrito` con el número de viajes (`n_viajes`) y la tarifa media (`tarifa_media`, 2 decimales) por `distrito`. Haz el join usando `F.broadcast(zonas)` y ordena por `n_viajes` de mayor a menor.

In [ ]:
display(por_distrito)

In [ ]:
# comprobación
assert por_distrito.columns == ["distrito", "n_viajes", "tarifa_media"]
assert por_distrito.agg(F.sum("n_viajes")).first()[0] == viajes_zona.count()
ok()

## Módulo 7. Funciones de ventana

Una ventana calcula un valor para cada fila mirando un grupo de filas relacionadas, pero sin colapsarlas en una sola como hace `groupBy`.

```python
w = Window.partitionBy("grupo").orderBy(F.desc("valor"))
df.withColumn("posicion", F.row_number().over(w))
```
Las más usadas son `row_number`, `rank`, `dense_rank`, `lag` y `lead`, y también agregaciones como `sum` sobre una ventana (para acumulados).

### Ejercicio 7.1: top 3 zonas por día de la semana
Crea `top3_por_dia` con los 3 `pickup_zip` con más viajes para cada `dia_semana`. Columnas: `dia_semana`, `pickup_zip`, `n_viajes`, `posicion`. El conteo por día y zip ya está hecho en `conteo`.

In [ ]:
conteo = viajes.groupBy("dia_semana", "pickup_zip").agg(F.count("*").alias("n_viajes"))


display(top3_por_dia)

In [ ]:
# comprobación
assert top3_por_dia.columns == ["dia_semana", "pickup_zip", "n_viajes", "posicion"]
assert top3_por_dia.groupBy("dia_semana").count().agg(F.max("count")).first()[0] <= 3
ok()

### Ejercicio 7.2: acumulado diario
Crea `por_fecha` con una fila por día y estas columnas:
- `fecha`: `F.to_date("tpep_pickup_datetime")`
- `n_viajes`: viajes de ese día
- `acumulado`: suma acumulada de `n_viajes` hasta ese día (incluido)

Las dos primeras ya están en `diario`, te falta el acumulado.

Pista: `Window.orderBy("fecha").rowsBetween(Window.unboundedPreceding, Window.currentRow)`

In [ ]:
diario = (viajes
    .groupBy(F.to_date("tpep_pickup_datetime").alias("fecha"))
    .agg(F.count("*").alias("n_viajes"))
)


display(por_fecha)

In [ ]:
# comprobación
ultima = por_fecha.orderBy(F.desc("fecha")).first()
assert ultima["acumulado"] == viajes.count(), "El último acumulado debe ser igual al total de viajes"
ok()

### Ejercicio 7.3: variación respecto al día anterior
A partir de `por_fecha`, crea `variacion_diaria` con la columna `variacion`, que es `n_viajes` menos los `n_viajes` del día anterior. Usa `F.lag`.

In [ ]:
display(variacion_diaria)

In [ ]:
# comprobación
filas = variacion_diaria.orderBy("fecha").collect()
assert filas[0]["variacion"] is None, "El primer día no tiene día anterior: debería ser null"
assert all(filas[i]["variacion"] == filas[i]["n_viajes"] - filas[i - 1]["n_viajes"] for i in range(1, len(filas)))
ok()

## Módulo 8. Spark SQL

Sobre los mismos datos se puede mezclar SQL y Python:

1. Registras el DataFrame como vista temporal con `df.createOrReplaceTempView("nombre")`
2. Consultas con `spark.sql("SELECT ...")`, que devuelve un DataFrame
3. O usas una celda que empiece con `%sql`

Las vistas temporales solo existen en tu sesión. En serverless no hay vistas temporales globales.

### Ejercicio 8.1: de DataFrame a SQL
Registra `viajes` como la vista temporal `viajes` y después, con `spark.sql`, crea `res_sql` con el número de viajes (`n_viajes`) por `tipo_viaje`.

In [ ]:
display(res_sql)

In [ ]:
# comprobación
a = {r["tipo_viaje"]: r["n_viajes"] for r in res_sql.collect()}
b = {r["tipo_viaje"]: r["n_viajes"] for r in por_tipo.collect()}
assert a == b, "El resultado en SQL debería coincidir con por_tipo del ejercicio 5.2"
ok()

### Ejercicio 8.2: celda `%sql`
En la celda de abajo escribe una consulta SQL directamente sobre `samples.nyctaxi.trips` que devuelva los 5 `pickup_zip` con mayor tarifa promedio (`tarifa_media`, redondeada a 2), tomando solo los zips con al menos 20 viajes.

Pista: `GROUP BY` + `HAVING COUNT(*) >= 20` + `ORDER BY ... DESC LIMIT 5`

In [ ]:
%sql
-- consulta sobre samples.nyctaxi.trips


### Ejercicio 8.3: SQL y DataFrames juntos
Con `spark.sql`, saca los viajes en hora punta de la mañana (`hora` entre 7 y 9, inclusive) desde la vista `viajes`. Luego, con la API de Python, agrupa por `tipo_viaje` contando los viajes (`n_viajes`). Guarda el resultado en `hora_punta`.

In [ ]:
display(hora_punta)

In [ ]:
# comprobación
assert hora_punta.agg(F.sum("n_viajes")).first()[0] == viajes.filter(F.col("hora").between(7, 9)).count()
ok()

## Módulo 9. Limpieza de datos

Para datos sucios tenemos:

| Problema | Qué usar |
|---|---|
| filas duplicadas | `dropDuplicates()` o `dropDuplicates(["col"])` |
| nulos | `fillna({"col": valor})`, `dropna(subset=[...])`, `F.col("c").isNull()` |
| espacios y mayúsculas | `F.trim`, `F.lower`, `F.upper`, `F.initcap` |
| texto que debería ser número | `F.expr("try_cast(col AS INT)")`, devuelve `null` si no se puede convertir |

Con ANSI activo, `F.col("edad").cast("int")` sobre un `"N/A"` lanza error, mientras que `try_cast` devuelve `null`.

In [ ]:
clientes_sucios = spark.createDataFrame(
    [
        (1, "  ana pérez ", "ANA@MAIL.COM", "34", "lima"),
        (2, "Luis Gómez", "luis@mail.com", None, "Bogotá"),
        (2, "Luis Gómez", "luis@mail.com", None, "Bogotá"),
        (3, "MARTA RUIZ", None, "29", "quito"),
        (4, "jorge díaz", "jorge@mail.com", "N/A", "Lima"),
        (5, None, "sofia@mail.com", "41", None),
    ],
    "id INT, nombre STRING, email STRING, edad STRING, ciudad STRING",
)
display(clientes_sucios)

### Ejercicio 9.1: nulos por columna
Crea `nulos`, un DataFrame de una sola fila con la cantidad de valores nulos de cada columna de `clientes_sucios`.

Pista: `F.col(c).isNull().cast("int")` vale 1 cuando el valor es nulo. Súmalo con `F.sum` para cada columna `c` usando una list comprehension, y ponle `alias(c)` para que cada columna conserve su nombre.

In [ ]:
display(nulos)

In [ ]:
# comprobación
fila = nulos.first()
assert nulos.columns == clientes_sucios.columns
assert (fila["id"], fila["nombre"], fila["email"], fila["edad"], fila["ciudad"]) == (0, 1, 1, 2, 1)
ok()

### Ejercicio 9.2: quitar duplicados
Crea `clientes_unicos` sin las filas duplicadas.

In [ ]:
print(clientes_unicos.count())

In [ ]:
# comprobación
assert clientes_unicos.count() == 5
ok()

### Ejercicio 9.3: normalizar y rellenar
A partir de `clientes_unicos`, crea `clientes_limpios` así:
- `nombre`: sin espacios de sobra y con mayúscula inicial en cada palabra (`"  ana pérez "` pasa a `"Ana Pérez"`)
- `email`: en minúsculas
- `edad`: convertida a entero con `try_cast` (`"N/A"` queda en `null`)
- `ciudad`: con mayúscula inicial
- nulos: `nombre` se rellena con `"Desconocido"` y `ciudad` con `"Sin ciudad"`

In [ ]:
display(clientes_limpios.orderBy("id"))

In [ ]:
# comprobación
c = {r["id"]: r for r in clientes_limpios.collect()}
assert dict(clientes_limpios.dtypes)["edad"] == "int", "edad debe ser entero"
assert c[1]["nombre"] == "Ana Pérez" and c[1]["email"] == "ana@mail.com" and c[1]["ciudad"] == "Lima"
assert c[4]["edad"] is None and c[3]["edad"] == 29
assert c[5]["nombre"] == "Desconocido" and c[5]["ciudad"] == "Sin ciudad"
ok()

### Ejercicio 9.4: descartar registros inválidos
A partir de `clientes_limpios`, crea `clientes_validos` quitando los clientes que no tienen email.

In [ ]:
display(clientes_validos)

In [ ]:
# comprobación
assert clientes_validos.count() == 4 and clientes_validos.filter(F.col("email").isNull()).count() == 0
ok()

## Módulo 10. Escribir tablas y archivos

En Databricks normalmente escribimos en dos lugares:

- Tablas de Unity Catalog (en formato Delta por defecto): `df.write.mode("overwrite").saveAsTable("catalogo.esquema.tabla")`
- Archivos dentro de un volumen, con rutas tipo `/Volumes/catalogo/esquema/volumen/carpeta`

Modos de escritura: `"overwrite"`, `"append"`, `"ignore"` y `"error"` (el de por defecto, falla si la tabla ya existe).

### Ejercicio 10.1: guardar como tabla Delta
Guarda `por_tipo` (del ejercicio 5.2) como la tabla `workspace.default.taxis_por_tipo`, sobrescribiendo si ya existe. Después léela de nuevo en `tabla_leida`.

In [ ]:
nombre_tabla = f"{CATALOGO}.{ESQUEMA}.taxis_por_tipo"


display(tabla_leida)

In [ ]:
# comprobación
assert tabla_leida.count() == por_tipo.count() == 3
ok("Correcto. La tabla aparece en Catalog > workspace > default")

Delta guarda un historial de cada escritura. Ejecuta esta celda para verlo; si vuelves a correr el 10.1 vas a ver una versión nueva.

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {CATALOGO}.{ESQUEMA}.taxis_por_tipo"))

### Ejercicio 10.2: CSV en un volumen
Primero creamos un volumen, que es una carpeta para archivos gobernada por Unity Catalog:

In [ ]:
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{ESQUEMA}.curso_pyspark")
RUTA_VOLUMEN = f"/Volumes/{CATALOGO}/{ESQUEMA}/curso_pyspark"
print(RUTA_VOLUMEN)

Ahora:
1. Escribe `top_zips` (ejercicio 5.3) como CSV con encabezado en `RUTA_VOLUMEN + "/top_zips"`, en modo overwrite.
2. Léelo de vuelta en `top_zips_leido`, con encabezado e inferencia de tipos.

In [ ]:
ruta_csv = RUTA_VOLUMEN + "/top_zips"


display(top_zips_leido)

In [ ]:
# comprobación
assert top_zips_leido.count() == top_zips.count()
assert set(top_zips_leido.columns) == {"pickup_zip", "n_viajes", "tarifa_media"}
ok()

Fíjate que Spark no escribe un único archivo sino una carpeta con uno o varios archivos `part-*` (uno por partición):

In [ ]:
display(dbutils.fs.ls(ruta_csv))

## Desafío final: ventas por país con TPC-H

Aquí toca juntar todo lo anterior. Las tablas de `samples.tpch` simulan un sistema de pedidos:

| Tabla | Columnas que vas a necesitar |
|---|---|
| `orders` | `o_orderkey`, `o_custkey`, `o_totalprice`, `o_orderdate` |
| `customer` | `c_custkey`, `c_nationkey` |
| `nation` | `n_nationkey`, `n_name`, `n_regionkey` |
| `region` | `r_regionkey`, `r_name` |

In [ ]:
orders   = spark.read.table("samples.tpch.orders")
customer = spark.read.table("samples.tpch.customer")
nation   = spark.read.table("samples.tpch.nation")
region   = spark.read.table("samples.tpch.region")
display(orders.limit(5))

### Desafío A: top 5 países por facturación en 1995
Crea `top_paises` con las columnas `pais`, `n_pedidos` y `facturacion` (suma de `o_totalprice`, redondeada a 2), solo con los pedidos cuyo `o_orderdate` sea de 1995. Ordena por facturación de mayor a menor y quédate con 5 filas.

Un orden posible: filtrar `orders` por año, join con `customer`, join con `nation`, agrupar, ordenar y limitar.

In [ ]:
display(top_paises)

In [ ]:
# comprobación
filas = top_paises.collect()
assert top_paises.columns == ["pais", "n_pedidos", "facturacion"] and len(filas) == 5
assert all(filas[i]["facturacion"] >= filas[i + 1]["facturacion"] for i in range(4))
ok("Desafío A listo")

### Desafío B (bonus): el mejor país de cada región
Considerando todos los años, crea `mejor_pais_region` con el país de mayor facturación total en cada región (`r_name`, que debe quedar como columna `region`). Columnas: `region`, `pais`, `facturacion`.

Idea: agrega por región y país, y después usa una ventana con `row_number` particionada por región.

In [ ]:
display(mejor_pais_region)

In [ ]:
# comprobación
assert mejor_pais_region.columns == ["region", "pais", "facturacion"]
assert mejor_pais_region.count() == region.count()
ok("Desafío B listo")

## Limpieza (opcional)
Si quieres borrar lo que creaste, descomenta estas líneas y ejecútalas:

In [ ]:
# spark.sql(f"DROP TABLE IF EXISTS {CATALOGO}.{ESQUEMA}.taxis_por_tipo")
# spark.sql(f"DROP VOLUME IF EXISTS {CATALOGO}.{ESQUEMA}.curso_pyspark")

## Para seguir
- Documentación de PySpark: https://spark.apache.org/docs/latest/api/python/
- Databricks Free Edition: https://docs.databricks.com/aws/en/getting-started/free-edition
- Databricks Academy tiene cursos gratuitos para seguir practicando.
- Lo que viene después: Structured Streaming, MLlib y pipelines declarativos (Lakeflow), que usan la misma API de DataFrames.